## Ejercicios

   1. Suma todos los cost_in_credits de TODOS los vehículos del archivo vehicles.csv. A continuación, suma los cost_in_credits por vehicle_class.
   
   2. Cuenta cuántos manufacturer distintos tenemos.
   
   3. Muestra la media y la desviación estándar de los passengers (de todos los vehículos) y después solo de los vehicle_class (repulsorcraft).
   
   4. Calcula el max_atmosphering_speed medio y la length media POR vehicle_class.
   
   5. Suma TODAS las cargo_capacity de TODOS los vehciulos en el DF. A continuación, suma TODOS los valores de crew por vehicle_class. ¿Ve valores nulos? ¿Por qué? ¿Cómo puede resolverlo?

In [ ]:
from pyspark.sql.functions import *

# ============================================================
# 0. Cargar el CSV
# ============================================================
vehicles = (
    spark.read
        .option("header", True)
        .option("inferSchema", True)
        .csv("vehicles.csv")
)

vehicles.show()


# ============================================================
# 1. Suma de cost_in_credits (total y por vehicle_class)
# ============================================================

# A) Suma total
suma_total = vehicles.agg(sum("cost_in_credits").alias("suma_total"))
suma_total.show()

# B) Suma por vehicle_class
suma_por_clase = (
    vehicles.groupBy("vehicle_class")
            .agg(sum("cost_in_credits").alias("suma_por_clase"))
)
suma_por_clase.show()


# ============================================================
# 2. Contar manufacturer distintos
# ============================================================

num_manufacturers = vehicles.select("manufacturer").distinct().count()
print("Número de manufacturers distintos:", num_manufacturers)


# ============================================================
# 3. Media y desviación estándar de passengers
# ============================================================

# A) De todos los vehículos
stats_passengers = vehicles.agg(
    mean("passengers").alias("media_passengers"),
    stddev("passengers").alias("std_passengers")
)
stats_passengers.show()

# B) Solo de vehicle_class == "repulsorcraft"
stats_repulsorcraft = (
    vehicles.filter(col("vehicle_class") == "repulsorcraft")
            .agg(
                mean("passengers").alias("media_passengers_repulsor"),
                stddev("passengers").alias("std_passengers_repulsor")
            )
)
stats_repulsorcraft.show()


# ============================================================
# 4. max_atmosphering_speed medio y length media POR vehicle_class
# ============================================================

vel_y_longitud = (
    vehicles.groupBy("vehicle_class")
            .agg(
                mean("max_atmosphering_speed").alias("vel_media"),
                mean("length").alias("longitud_media")
            )
)
vel_y_longitud.show()


# ============================================================
# 5. Suma cargo_capacity total y suma crew POR vehicle_class
# ============================================================

# A) Suma total de cargo_capacity
suma_cargo = vehicles.agg(sum("cargo_capacity").alias("cargo_total"))
suma_cargo.show()

# B) Suma crew por vehicle_class
crew_por_clase = (
    vehicles.groupBy("vehicle_class")
            .agg(sum("crew").alias("crew_total"))
)
crew_por_clase.show()

# Preguntas:
# ¿Ves valores nulos? ¿Por qué?
# - Algunos vehículos tienen crew = null.
#   Esto provoca resultados null al sumar.

# ¿Cómo resolverlo?
# Usar fillna o coalesce para reemplazar null por 0.
crew_por_clase_sin_nulls = (
    vehicles
    .withColumn("crew", coalesce(col("crew"), lit(0)))
    .groupBy("vehicle_class")
    .agg(sum("crew").alias("crew_total_limpio"))
)
crew_por_clase_sin_nulls.show()
